In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim, nn
from torch.utils.data import DataLoader
from tqdm import tqdm

import torchvision 
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import torchmetrics 



# Things to change about:
- Preprocessing:
  - Grayscale
  - Gaussian Blur
- Architectrue:
  - Kernel size
  - Pooling size
  - 
- Training:
  - Learning rate
  - Find min loss params?

In [2]:
batch_size = 5

transform = transforms.Compose([
    transforms.Resize((20,20)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder('./cnnTraining/training_data', transform= transform)
test_dataset = datasets.ImageFolder('./cnnTraining/testing_data', transform= transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)


In [3]:
class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()

        kernelOneSize =3
        kernelTwoSize = 3

        out1 =6
        out2 = 16

        self.conv1 = nn.Conv2d(in_channels=3, out_channels= out1, kernel_size= kernelOneSize)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels= out1, out_channels= out2, kernel_size= kernelTwoSize)
        self.dense1 = nn.Linear(in_features= out2*kernelTwoSize*kernelTwoSize, out_features=120)
        self.dense2 = nn.Linear(in_features=120,out_features=84)
        self.dense3 = nn.Linear(in_features=84, out_features=26)
        

    def forward(self,imgs):
        conv1Out = self.conv1(imgs)
        conv2In = self.pool( F.relu(conv1Out) )
        conv2Out = self.conv2(conv2In)
        dense1In = self.pool( F.relu(conv2Out) )
        dense1In = dense1In.view(-1, 16*3*3) #Currently hardcoded
        dense1Out = F.relu( self.dense1(dense1In) )
        dense2Out = F.relu( self.dense2(dense1Out) )
        dense3Out = self.dense3(dense2Out)
        return dense3Out



In [7]:
class CNNRunner():
    def __init__(self, net: Net, lossFunc, optimizer:optim):
        self.net = net
        self.lossFunc = lossFunc
        self.optim = optimizer

    def runEpoch(self, dataloader:DataLoader, reportInterval = 50):
        totalLoss = 0.0
        for i, data in enumerate(dataloader,0):
            inputs, labels = data
            self.optim.zero_grad()

            outputs = self.net(inputs)
            loss = self.lossFunc(outputs,labels)
            loss.backward()
            self.optim.step()
            totalLoss += loss.item()

            if i % reportInterval == reportInterval-1: 
                print(f'    {i+1} samples - Avg loss: {totalLoss/i} ')

        return totalLoss

    def train(self, epochs:int, dataLoader:DataLoader):
        for epoch in range(epochs):
            print(f'Epoch {epoch+1}:')
            self.runEpoch(dataLoader)

    def test(self, testLoader:DataLoader):
        correct = 0
        total = 0
        with torch.no_grad():
            for data in testLoader:
                images, labels = data
                outputs = self.net(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        print(f'Total: {total}\nCorrect: {correct}\nAccuracy: {round(100*(correct/total),2)}')




In [8]:
net = Net()

lossFunc = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

runner = CNNRunner(net,lossFunc,optimizer)

runner.train(5,train_loader)
runner.test(test_loader)

Epoch 1:
    50 samples - Avg loss: 3.329909188406808 
    100 samples - Avg loss: 3.2870686704462226 
    150 samples - Avg loss: 3.2338426929192257 
    200 samples - Avg loss: 3.053611654133054 
Epoch 2:
    50 samples - Avg loss: 1.4680439294600973 
    100 samples - Avg loss: 1.2553296311937197 
    150 samples - Avg loss: 1.1076694556930722 
    200 samples - Avg loss: 1.0001475530952664 
Epoch 3:
    50 samples - Avg loss: 0.5927124979848765 
    100 samples - Avg loss: 0.5288108659889361 
    150 samples - Avg loss: 0.48276766650253694 
    200 samples - Avg loss: 0.43778875347077095 
Epoch 4:
    50 samples - Avg loss: 0.3940998239594759 
    100 samples - Avg loss: 0.3077399241596912 
    150 samples - Avg loss: 0.27545831227462564 
    200 samples - Avg loss: 0.260315116485534 
Epoch 5:
    50 samples - Avg loss: 0.08266623962162138 
    100 samples - Avg loss: 0.11137478421101196 
    150 samples - Avg loss: 0.1349339489868042 
    200 samples - Avg loss: 0.1597043616172445